In [0]:
from pyspark.sql import functions as F

bronze_df = spark.table(
    "workspace.default.capstone_bronze_sales"
)

silver_df = bronze_df.select(
    F.col("order_id"),
    F.to_date("order_date").alias("order_date"),
    F.trim(F.col("customer_id")).alias("customer_id"),
    F.trim(F.col("customer_name")).alias("customer_name"),
    F.trim(F.col("city")).alias("city"),
    F.trim(F.col("state")).alias("state"),
    F.trim(F.col("product_id")).alias("product_id"),
    F.trim(F.col("product_name")).alias("product_name"),
    F.trim(F.col("category")).alias("category"),
    F.col("quantity").cast("int").alias("quantity"),
    F.col("unit_price").cast("double").alias("unit_price"),
    F.col("discount_pct").cast("double").alias("discount_pct"),
    F.col("gross_amount").cast("double").alias("gross_amount"),
    F.col("discount_amount").cast("double").alias("discount_amount"),
    F.col("net_amount").cast("double").alias("net_amount"),
    F.trim(F.col("payment_method")).alias("payment_method"),
    F.trim(F.col("order_status")).alias("order_status")
)

# Null handling
silver_df = silver_df.fillna({
    "customer_name": "UNKNOWN",
    "city": "UNKNOWN",
    "state": "UNKNOWN",
    "category": "UNKNOWN"
})

# Data Quality Flag
silver_df = silver_df.withColumn(
    "quality_flag",
    F.when(
        F.col("customer_id").isNull(),
        "INVALID_CUSTOMER"
    )
    .when(
        F.col("quantity") <= 0,
        "INVALID_QUANTITY"
    )
    .when(
        F.col("net_amount") < 0,
        "INVALID_AMOUNT"
    )
    .when(
        F.upper(F.col("product_id")) == "UNKNOWN",
        "INVALID_PRODUCT"
    )
    .otherwise("VALID")
)

# Keep only valid records
silver_df = silver_df.filter(
    F.col("quality_flag") == "VALID"
)

# Date attributes
silver_df = silver_df.withColumn(
    "year",
    F.year("order_date")
).withColumn(
    "month",
    F.month("order_date")
).withColumn(
    "month_name",
    F.date_format("order_date", "MMMM")
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.capstone_silver_sales")

print("Silver table created successfully")

In [0]:
%sql
select count(*) as total_recods from workspace.default.capstone_silver_sales;
